#Session 07
Built-in Functions `pyspark.sql.functions`
**Topics:** String · Date · Conditional · Array

---
## 1 · String Functions
`upper()` `lower()` `trim()` `concat()` `split()` `regexp_replace()`

In [0]:
import pyspark.sql.functions as F

In [0]:
employees = [
    (1, "  alice sharma  ", "alice@company.com", "9876543210"),
    (2, "BOB KUMAR",        "bob@company.com",   "9123456780"),
    (3, "Charlie Das",      "charlie@co.in",     "9000011111"),
    (4, "diana NAIR",       "diana@company.com", "9555566666"),
]
emp_df = spark.createDataFrame(employees, ["id", "name", "email", "phone"])
emp_df.show(truncate=False)

In [0]:
# upper(), lower(), trim()
emp_df.select(
    "name",
    F.upper("name").alias("upper_name"),
    F.lower("name").alias("lower_name"),
    F.trim(F.lower("name")).alias("clean_name")
).show(truncate=False)

In [0]:
# concat() — combine columns or literal strings
emp_df.select(
    F.concat(F.trim(F.lower("name")), F.lit(" | "), "email").alias("name_email")
).show(truncate=False)

In [0]:
# split() — splits a string into an array
emp_df.select(
    "email",
    F.split("email", "@").alias("email_parts"),
    F.split("email", "@").getItem(0).alias("username"),
    F.split("email", "@").getItem(1).alias("domain")
).show(truncate=False)

In [0]:
# regexp_replace(column, pattern, replacement)
emp_df.select(
    "phone",
    F.regexp_replace("phone", r"(\d{5})(\d{5})", "XXXXX$2").alias("masked_phone")
).show()

#https://www.w3schools.com/python/python_regex.asp

---
## 2 · Date Functions
`current_date()` `to_date()` `datediff()` `date_format()`

In [0]:
orders = [
    (101, "2024-01-15", "2024-02-10"),
    (102, "2024-03-22", "2024-04-05"),
    (103, "2023-11-01", "2023-11-20"),
    (104, "2024-06-30", "2024-07-15"),
]
orders_df = spark.createDataFrame(orders, ["order_id", "order_date", "delivery_date"])
orders_df.printSchema()

In [0]:
# Strings are not dates yet — cast them using to_date()
orders_df = orders_df.withColumn("order_date",    F.to_date("order_date",    "yyyy-MM-dd")) \
                     .withColumn("delivery_date", F.to_date("delivery_date", "yyyy-MM-dd"))
orders_df.printSchema()

# cast() works only when the source is in same yyyy-MM-dd format. 

In [0]:
# current_date() and datediff()
orders_df.select(
    "order_id",
    "order_date",
    "delivery_date",
    F.current_date().alias("today"),
    F.datediff("delivery_date", "order_date").alias("days_to_deliver"),
    F.datediff(F.current_date(), "order_date").alias("days_since_order")
).show()

In [0]:
# date_format() — format a date column as a readable string
orders_df.select(
    "order_id",
    F.date_format("order_date", "dd MMM yyyy").alias("formatted_order"),
    F.date_format("order_date", "MMMM").alias("month_name"),
    F.date_format("order_date", "EEEE").alias("day_of_week")
).show()

---
## 3 · Conditional Logic
`when().otherwise()`

In [0]:
sales = [
    (1, "Alice",   92000, "Chennai"),
    (2, "Bob",     45000, "Mumbai"),
    (3, "Charlie", 67000, "Bangalore"),
    (4, "Diana",   31000, "Chennai"),
    (5, "Eve",     78000, "Delhi"),
    (6, "Frank",   55000, "Mumbai"),
]
sales_df = spark.createDataFrame(sales, ["id", "name", "salary", "city"])
sales_df.show()

In [0]:
# Single condition — F.when(condition, value).otherwise(value)
sales_df.withColumn(
    "grade",
    F.when(F.col("salary") >= 70000, "Senior").otherwise("Junior")
).show()

In [0]:
# Chained conditions — like if / elif / else
sales_df.withColumn(
    "band",
    F.when(F.col("salary") >= 80000, "Band A")
     .when(F.col("salary") >= 60000, "Band B")
     .when(F.col("salary") >= 40000, "Band C")
     .otherwise("Band D")
).show()

In [0]:
# Condition on multiple columns
sales_df.withColumn(
    "flag",
    F.when(
        (F.col("salary") >= 70000) & (F.col("city") == "Chennai"),
        "High Earner - Chennai"
    ).when(
        F.col("salary") >= 70000, 
        "High Earner - Other"
    ).otherwise("Standard")
).show(truncate=False)

---
## 4 · Array Functions
`explode()` `collect_list()` `array_contains()`

In [0]:
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, IntegerType

# Each employee has a list of skills
schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("name",   StringType()),
    StructField("skills", ArrayType(StringType()))
])

data = [
    (1, "Alice",   ["Python", "Spark", "SQL"]),
    (2, "Bob",     ["Java", "Scala", "Spark"]),
    (3, "Charlie", ["Python", "SQL"]),
    (4, "Diana",   ["Spark", "Azure", "Python"]),
]
skills_df = spark.createDataFrame(data, schema)
skills_df.show(truncate=False)

In [0]:
# explode() — one row per array element
skills_df.select(
    "emp_id", "name",
    F.explode("skills").alias("skill")
).show()

In [0]:
# array_contains(column, value) — filter or flag rows
skills_df.withColumn(
    "knows_spark", F.array_contains("skills", "Spark")
).show(truncate=False)

In [0]:
# collect_list() — inverse of explode: aggregate rows back into an array
# First, create a flat table (as if we got it from a database)
flat_data = [
    (1, "Alice",   "Python"),
    (1, "Alice",   "Spark"),
    (1, "Alice",   "SQL"),
    (2, "Bob",     "Java"),
    (2, "Bob",     "Scala"),
    (3, "Charlie", "Python"),
    (3, "Charlie", "SQL"),
]
flat_df = spark.createDataFrame(flat_data, ["emp_id", "name", "skill"])
flat_df.show()

In [0]:
# collect_list() groups individual rows into an array per key
flat_df.groupBy("emp_id", "name") \
       .agg(F.collect_list("skill").alias("skills")) \
       .orderBy("emp_id") \
       .show(truncate=False)

---
## Quick Reference

| Category | Function | Purpose |
|----------|----------|---------|
| String | `upper()` `lower()` `trim()` | Case & whitespace |
| String | `concat()` | Join multiple columns/literals |
| String | `split(col, pattern)` | String → Array |
| String | `regexp_replace(col, pattern, repl)` | Regex substitution |
| Date | `to_date(col, format)` | String → Date |
| Date | `current_date()` | Today's date |
| Date | `datediff(end, start)` | Days between two dates |
| Date | `date_format(col, pattern)` | Date → formatted String |
| Conditional | `when(cond, val).otherwise(val)` | if / elif / else |
| Array | `explode(col)` | Array → multiple rows |
| Array | `collect_list(col)` | Rows → Array (groupBy) |
| Array | `array_contains(col, val)` | Check if value exists in array |